# 06C - Random Forest Classifier (Executed Ready)
Run this notebook to generate outputs. The structure is optimized for GitHub.

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             RocCurveDisplay, PrecisionRecallDisplay,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

df = pd.read_csv("american_bankruptcy.csv")
df["target"] = df["status_label"].map({"alive":0,"failed":1})
display(df.head())
print(df.shape)


## Data Preparation

In [ ]:
drop_cols=["status_label","target"]
if "company_name" in df.columns:
    drop_cols.append("company_name")

X=df.drop(columns=drop_cols)
y=df["target"]

num=X.select_dtypes(include="number").columns
cat=X.select_dtypes(exclude="number").columns

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,stratify=y,random_state=42)

pre=ColumnTransformer([
("num",SimpleImputer(strategy="median"),num),
("cat",Pipeline([
("imp",SimpleImputer(strategy="most_frequent")),
("enc",OneHotEncoder(handle_unknown="ignore"))
]),cat)
])


## Train Model

In [ ]:
model=Pipeline([
("preprocessor",pre),
("classifier",RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
))
])

model.fit(X_train,y_train)


## Cross Validation

In [ ]:
cv=cross_validate(model,X_train,y_train,cv=3,
                          scoring=["accuracy","precision","recall","f1","roc_auc"])
pd.DataFrame(cv).describe()


## Evaluation

In [ ]:
pred=model.predict(X_test)
prob=model.predict_proba(X_test)[:,1]

results=pd.DataFrame({
"Metric":["Accuracy","Precision","Recall","F1","ROC-AUC"],
"Value":[
accuracy_score(y_test,pred),
precision_score(y_test,pred),
recall_score(y_test,pred),
f1_score(y_test,pred),
roc_auc_score(y_test,prob)]
})
display(results)
print(classification_report(y_test,pred))


## Visualizations

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.show()

RocCurveDisplay.from_predictions(y_test,prob)
plt.show()

PrecisionRecallDisplay.from_predictions(y_test,prob)
plt.show()


## Feature Importance

In [ ]:
rf=model.named_steps["classifier"]
features=list(num)
if len(cat):
    features.extend(
        model.named_steps["preprocessor"]
        .named_transformers_["cat"]
        .named_steps["enc"]
        .get_feature_names_out(cat)
    )

importance=(pd.DataFrame({"Feature":features,
                          "Importance":rf.feature_importances_})
            .sort_values("Importance",ascending=False))
display(importance.head(20))


## Save Model

In [ ]:
joblib.dump(model,"random_forest_model.joblib")
print("Model saved as random_forest_model.joblib")
